In [4]:
%load_ext autoreload
%autoreload 2

import glob
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm

from tpa.utils.utils import *
from tpa.utils.io import *

setup_logging()
sns.set_style("darkgrid") 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
def load_bundle(bundle):
    bundle_dir="/ifs/loni/faculty/thompson/four_d/wfeng/Datasets/Atlas/ENIGMA_tract_atlas_v2/bundles_downsampled"
    streamlines, _  = load_streamlines(f"{bundle_dir}/{bundle}.trk")
    return streamlines

In [8]:
lines = load_bundle("AF_L")
len(lines)

6982

In [26]:
from scipy.spatial import cKDTree

from nibabel.streamlines.array_sequence import ArraySequence
from dipy.tracking.streamline import set_number_of_points, Streamlines, values_from_volume
from dipy.tracking.utils import length
from dipy.segment.metric import AveragePointwiseEuclideanMetric
from dipy.segment.clustering import QuickBundles

def bundle_centroid(bundle, n_points=100, thresh=85):
    if isinstance(bundle, np.ndarray):
        bundle = ArraySequence(bundle[:,:,:3])
        
    ref_bundle = set_number_of_points(bundle, nb_points=n_points)
    metric = AveragePointwiseEuclideanMetric()
    qb = QuickBundles(threshold=thresh, metric=metric)
    clusters = qb.cluster(ref_bundle)
    centroids = Streamlines(clusters.centroids)
    return centroids

def buan_segment_idx(bundle, ref_bundle, n_segments=100):
    '''
        Create BUAN segments from a reference bundle
        Returns distance to the nearest point on the centroid and 
        segment index for each point on the bundle
    '''
    if isinstance(bundle, np.ndarray):
        bundle = ArraySequence(bundle[:,:,:3])

    centroid = bundle_centroid(ref_bundle, n_points=n_segments, thresh=85)
    dist, indx = cKDTree(centroid[0], 1, copy_data=True).query(
        bundle.get_data(), k=1
    )
    return centroid, dist, indx


def buan_segment_metric(rec_bundle, org_bundle, ref_bundle, 
                       scalar_map, scalar_affine, 
                       n_segments=100, epsilon=1e-8):
    '''Compute along-tract weighted mean profile using BUAN'''
    
    # Compute weights (based on distance) and segments indices
    dist, indx = buan_segment_idx(rec_bundle, ref_bundle, n_segments)
    weights = 1 / (dist + epsilon)
    
    # Project scalar map to bundle
    values = values_from_volume(scalar_map, org_bundle, scalar_affine)
    values_flattened = np.array([val for values_line in values for val in values_line])
    
    # Compute weighted mean
    profile = []
    for i in range(n_segments):
        segment_mask = (indx==i) & (~np.isnan(values_flattened)) #np.where(indx==i)[0]
        if len(segment_mask)>1:
            vals = values_flattened[segment_mask]
            wts = weights[segment_mask]
            wts /= np.sum(wts)
            weighted_mean = np.sum(wts * vals)
            profile.append(weighted_mean)
        else:
            profile.append(0)
    return profile

In [29]:
centroid, dist, indx = buan_segment_idx(lines, lines, n_segments=10)

In [45]:
import numpy as np
from scipy.spatial import cKDTree

def assign_points_to_tract_segments(bundle_points, centroid):
    """
    Assign each point in the bundle to along-tract coordinate (s)
    using curvature-aware segment projection.

    Parameters
    ----------
    bundle_points : (M, 3) array
        All points from all streamlines.
    centroid : (N, 3) array
        Centroid polyline with N segments (ordered along tract).
    return_radial : bool
        Whether to return radial distances.

    Returns
    -------
    s_index : (M,) array
        Segment indices (0..N-1)
    s_dist : (M,) array
        Along-tract distances (mm)
    r_dist : (M,) array, optional
        Radial distances to centroid.
    """

    # --- Precompute arc lengths along centroid ---
    seg_vecs = np.diff(centroid, axis=0)
    seg_len = np.linalg.norm(seg_vecs, axis=1)
    s_centroid = np.concatenate([[0], np.cumsum(seg_len)])
    midpoints = 0.5 * (centroid[:-1] + centroid[1:])

    # --- Use KDTree on midpoints for initial guess ---
    tree = cKDTree(midpoints)
    _, nearest_seg = tree.query(bundle_points, k=1)

    s_dist = np.zeros(len(bundle_points))
    r_dist = np.zeros(len(bundle_points))
    s_index = np.zeros(len(bundle_points), dtype=int)

    # --- Local projection onto nearest segment ---
    for i, (x, seg_i) in enumerate(zip(bundle_points, nearest_seg)):
        p0, p1 = centroid[seg_i], centroid[seg_i + 1]
        v = p1 - p0
        t = np.dot(x - p0, v) / np.dot(v, v)
        t = np.clip(t, 0, 1)
        proj = p0 + t * v

        s_dist[i] = s_centroid[seg_i] + t * seg_len[seg_i]
        s_index[i] = seg_i
        r_dist[i] = np.linalg.norm(x - proj)

    return s_index, s_dist, r_dist

In [46]:
centroid = bundle_centroid(lines, n_points=10, thresh=85)
centroid[0].shape

(10, 3)

In [47]:
s_index, s_dist, r_dist = assign_points_to_tract_segments(lines.get_data(), centroid[0])

In [51]:
s_dist.shape

(1396400,)